# Notebook 06 — Gating Design for the Mixture-of-Experts System

This notebook is step 6 of the 9-notebook pipeline of the Mixture-of-Experts (MoE) forecaster for the hourly electricity demand of Colombia's National Interconnected System (SIN):

01 data acquisition (XM S.A. SINERGOX API) → 02 preprocessing and feature engineering → 03 clustering of daily and weekly demand profiles → 04 cluster analysis and cluster classifiers → 05 expert selection → **06 gating design** → 07 hyperparameter optimization (HPO) → 08 production assembly → 09 benchmark.

In an MoE forecaster the *gate* (router) decides, for each hourly observation, which expert or which weighted combination of experts produces the forecast. At forecast time the true day type and week type are unknown, so the gate must rely on the cluster classifiers of Notebook 04 or on the similarity to the cluster centroids of Notebook 03.

---

## Objective

Design and compare four gating (routing) mechanisms on a held-out test set and select one:

1. **Hard gating** — each observation is routed to the single expert of the predicted `(day type, week type)` pair (baseline).
2. **Soft gating, distance-based** — experts are combined with weights obtained from a temperature-scaled softmax of the distances to the cluster centroids.
3. **Soft gating, probability-based** — experts are combined with weights given by the joint class probabilities of the day and week classifiers.
4. **Adaptive gating** — hard gating when the classifiers are confident, probability-based soft gating otherwise.

The hyperparameters of the gates (temperature $\tau$, confidence threshold $\theta$) are explored on fixed grids, so that the trade-off between simplicity (hard routing) and robustness to misclassification (soft routing) can be assessed.

## Inputs

| File | Produced by | Content |
|:---|:---|:---|
| `DATA_DIR/processed/features.parquet` | Notebook 02 | Hourly demand and engineered features |
| `DATA_DIR/clustering/cluster_assignments.parquet` | Notebook 03 | Hourly `datetime` mapped to `cluster_day` and `cluster_week` (renamed here to `type_day`, `type_week`) |
| `DATA_DIR/clustering/centroids_day.pkl`, `centroids_week.pkl` | Notebook 03 | Dictionaries whose key `centroids` holds the day/week cluster centroids |
| `MODELS_DIR/classifiers/best_classifier_day.pkl`, `best_classifier_week.pkl` | Notebook 04 | Dictionaries whose key `pipeline` holds the fitted day/week classifiers (with `predict_proba`) |
| `MODELS_DIR/experts/expert_day{i}_week{j}.pkl` | Notebook 05 | One fitted expert per cluster combination |

## Outputs

| File | Content |
|:---|:---|
| `RESULTS_DIR/06_gating_comparison.csv` | Test-set metrics of every gating configuration evaluated |
| `RESULTS_DIR/06_best_per_method.csv` | Lowest-MAPE configuration of each gating method |
| `RESULTS_DIR/06_adaptive_threshold_sweep.csv` | Adaptive gating: metrics and hard/soft split for each threshold |
| `RESULTS_DIR/06_distance_tau_sweep.csv` | Distance-based soft gating: metrics for each temperature |
| `MODELS_DIR/gating/06_best_gating_config.json` | Selected gating configuration and reference metrics |
| `RESULTS_DIR/06_*.png` | Figures of Sections 6–9 |

The gating hyperparameters used by the production model are tuned again, jointly with the experts, in Notebook 07; the JSON file above is not read by Notebooks 07–09 and documents the outcome of this design study.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

---
## 1. Configuration

Random seed and the search grids of the gating hyperparameters:

- `THRESHOLD_SEARCH`: confidence thresholds $\theta$ for adaptive gating.
- `TAU_SEARCH`: softmax temperatures $\tau$ for distance-based soft gating.
- `MIN_PROB`: minimum joint probability for an expert to take part in probability-based soft gating.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS = -1

# Gating-specific search grids
THRESHOLD_SEARCH = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
TAU_SEARCH = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
MIN_PROB = 0.05  # minimum joint probability for an expert to be included in soft gating

import os, warnings
warnings.filterwarnings("ignore")

os.makedirs(f"{RESULTS_DIR}/gating", exist_ok=True)

print("Configuration loaded.")
print(f"  DATA_DIR    = {DATA_DIR}")
print(f"  MODELS_DIR  = {MODELS_DIR}")
print(f"  RESULTS_DIR = {RESULTS_DIR}")

---
## 2. Loading features, cluster assignments, classifiers and experts

All artifacts produced by Notebooks 02–05 are loaded (see the Inputs table above). If a required artifact is missing, the notebook stops with an error: no synthetic substitute data are generated.

### 2.1 Feature matrix

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import softmax as scipy_softmax

np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------
# Load the feature matrix
# ------------------------------------------------------------------
features_path = Path(os.path.join(DATA_DIR, "processed", "features.parquet"))
if features_path.exists():
    df_features = pd.read_parquet(features_path)
    # Make 'datetime' a column rather than the index
    if 'datetime' in df_features.index.names or df_features.index.name == 'datetime':
        df_features = df_features.reset_index()
    print(f"Features loaded: {df_features.shape}")
else:
    # No synthetic fallback data: if an artifact of a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{features_path} does not exist. Run Notebook 02 first (make pipeline).")

In [ ]:
df_features.head()

### 2.2 Cluster assignments

The hourly cluster assignments are loaded and their columns `cluster_day` / `cluster_week` are renamed to `type_day` / `type_week`. They are joined to the feature matrix by `datetime`, after which every row with a missing value is dropped. The number of observations of each `(type_day, type_week)` combination is printed.

In [ ]:
# ------------------------------------------------------------------
# Load the cluster assignments
# ------------------------------------------------------------------
clustering_dir = Path(DATA_DIR) / "clustering"
assignments_path = clustering_dir / "cluster_assignments.parquet"

if assignments_path.exists():
    df_assignments = pd.read_parquet(assignments_path)
    # Rename columns to 'type_day' and 'type_week'
    if 'cluster_day' in df_assignments.columns and 'cluster_week' in df_assignments.columns:
        df_assignments = df_assignments.rename(columns={
            'cluster_day': 'type_day',
            'cluster_week': 'type_week'
        })
    print(f"Cluster assignments loaded: {df_assignments.shape}")
    # Diagnostic print of the columns after loading/renaming
    print(f"  df_assignments columns after loading/renaming: {df_assignments.columns.tolist()}")
else:
    # No synthetic fallback data: if an artifact of a previous notebook is
    # missing, that notebook must be run first.
    raise FileNotFoundError(f"{assignments_path} does not exist. Run Notebook 03 first (make pipeline).")

# Join the cluster labels to the features if not already present
if "type_day" not in df_features.columns:
    if "date" not in df_features.columns:
        df_features["date"] = df_features["datetime"].dt.date
    df_features = df_features.merge(
        df_assignments[["datetime", "type_day", "type_week"]],
        on="datetime", how="left"
    )

# Drop rows with NaN once all joins and derived columns are in place
df_features = df_features.dropna().reset_index(drop=True)

# Cluster combinations
day_labels = sorted(df_features["type_day"].unique())
week_labels = sorted(df_features["type_week"].unique())
cluster_combos = [(d, w) for d in day_labels for w in week_labels]
print(f"\nCluster combinations (type_day x type_week): {len(cluster_combos)}")
for d, w in cluster_combos:
    n = len(df_features[(df_features["type_day"] == d) & (df_features["type_week"] == w)])
    print(f"  day={d}, week={w}: {n:,} observations")

In [ ]:
df_features.head()

### 2.3 Exclusion of 2020 and 2021

As in Notebook 05, observations from the years 2020 and 2021 are excluded.

In [ ]:
# Exclude the years 2020 and 2021. The index was reset above, so the year comes
# from the datetime column. (The cluster assignments from notebook 03 already
# cover only 2022 onwards, so after the merge and dropna this is a safeguard.)
df_features = df_features[~df_features["datetime"].dt.year.isin([2020, 2021])].reset_index(drop=True)

### 2.4 Feature set

The same feature set as in Notebook 05 is used: cyclical calendar encodings (hour, day of week, month), holiday indicators (`is_holiday`, `is_bridge` for bridge holidays, and the distances in days to the next/previous holiday) and all demand lags (`demand_lag_*`).

In [ ]:
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df_features.columns)

# Demand lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

In [ ]:
len(calendar_lags)

### 2.5 Train/test split and cluster classifiers

The data are split chronologically: the first 80 % of the rows form the training part and the last 20 % the **test set** on which every gating mechanism is evaluated. The day and week classifiers of Notebook 04 are loaded; if the number of features expected by a loaded classifier differs from the current feature set, the classifier is discarded and a warning is printed.

In [ ]:
import pickle
from pathlib import Path

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

clf_day_path = Path(MODELS_DIR) / "classifiers" / "best_classifier_day.pkl"
clf_week_path = Path(MODELS_DIR) / "classifiers" / "best_classifier_week.pkl"

# Feature columns
META_COLS = ["datetime", "date", "demand", "type_day", "type_week"]
# Calendar features + demand lags (Section 2.4)
FEATURE_COLS = calendar_lags
print(f"\nFeature columns ({len(FEATURE_COLS)}): {FEATURE_COLS[:10]}...")

X_all = df_features[FEATURE_COLS].values
y_all = df_features["demand"].values
y_day = df_features["type_day"].values.astype(int)
y_week = df_features["type_week"].values.astype(int)

# Chronological train/test split: last 20% of rows for testing
split_idx = int(len(df_features) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
y_day_train, y_day_test = y_day[:split_idx], y_day[split_idx:]
y_week_train, y_week_test = y_week[:split_idx], y_week[split_idx:]

df_test = df_features.iloc[split_idx:].copy().reset_index(drop=True)
print(f"\nTraining: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

if clf_day_path.exists() and clf_week_path.exists():
    with open(clf_day_path, "rb") as f:
        loaded_day_dict = pickle.load(f)
    with open(clf_week_path, "rb") as f:
        loaded_week_dict = pickle.load(f)
    # Extract the pipeline object from the loaded dictionary
    classifier_day = loaded_day_dict['pipeline']
    classifier_week = loaded_week_dict['pipeline']

    # Check feature-count compatibility; discard the loaded classifier if it does not match
    if hasattr(classifier_day, 'steps') and len(classifier_day.steps) > 0 and hasattr(classifier_day.steps[0][1], 'n_features_in_'):
        if classifier_day.steps[0][1].n_features_in_ != len(FEATURE_COLS):
            print("[WARNING] The loaded classifier_day expects a different number of features. Retraining is required.")
            classifier_day = None  # invalidate the loaded model
    if hasattr(classifier_week, 'steps') and len(classifier_week.steps) > 0 and hasattr(classifier_week.steps[0][1], 'n_features_in_'):
        if classifier_week.steps[0][1].n_features_in_ != len(FEATURE_COLS):
            print("[WARNING] The loaded classifier_week expects a different number of features. Retraining is required.")
            classifier_week = None  # invalidate the loaded model

    if classifier_day is not None and classifier_week is not None:
        print("Classifiers loaded from disk.")

### 2.6 Expert models

The experts of Notebook 05 (`expert_day{i}_week{j}.pkl`) are loaded and indexed by the pair `(i, j)`. Only if no expert file exists, a fallback set of experts is trained here: one `GradientBoostingRegressor` per combination with at least 30 training observations, fitted on the training part of the split, and saved to `MODELS_DIR/experts/`.

In [ ]:
# ------------------------------------------------------------------
# Load (or, if absent, train) the expert models
# ------------------------------------------------------------------
from sklearn.ensemble import GradientBoostingRegressor

experts_dir = Path(MODELS_DIR) / "experts"

experts = {}
existing_experts = list(experts_dir.glob("expert_day*_week*.pkl"))

if existing_experts:
    for path in existing_experts:
        name = path.stem
        parts = name.replace("expert_day", "").split("_week")
        cd, cw = int(parts[0]), int(parts[1])
        with open(path, "rb") as f:
            experts[(cd, cw)] = pickle.load(f)
    print(f"\nExperts loaded from disk: {len(experts)} models")
else:
    print("\n[INFO] No expert models found. Training fallback GradientBoosting experts.")
    for d_label in day_labels:
        for w_label in week_labels:
            mask_train = (y_day_train == d_label) & (y_week_train == w_label)
            n_train_cluster = mask_train.sum()
            if n_train_cluster < 30:
                print(f"  Skipping day={d_label}, week={w_label}: only {n_train_cluster} training observations")
                continue

            model = GradientBoostingRegressor(
                n_estimators=200, max_depth=5, learning_rate=0.1,
                random_state=RANDOM_SEED
            )
            model.fit(X_train[mask_train], y_train[mask_train])
            experts[(d_label, w_label)] = model

            # Save
            path = experts_dir / f"expert_day{d_label}_week{w_label}.pkl"
            with open(path, "wb") as f:
                pickle.dump(model, f)
            print(f"  Expert day={d_label}, week={w_label}: trained on {n_train_cluster:,} observations")

    print(f"  Total experts trained: {len(experts)}")

print(f"\nAvailable expert keys: {sorted(experts.keys())}")

---
## 3. Metric functions

All metrics are defined in this notebook so that it is self-contained. In this notebook MAPE is expressed **in percent** (unlike Notebook 05, where it is a fraction). The dictionary keys `"MAPE (%)"`, `"RMSE (kWh)"` and `"MAE (kWh)"` become column names of the saved CSV files.

In [ ]:
# ============================================================
# Metrics — fully self-contained, no external metric imports
# ============================================================

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error, in percent."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_metric(y_true, y_pred):
    """Mean Absolute Error."""
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def compute_all_metrics(y_true, y_pred):
    """Compute the main metrics and return them as a dictionary."""
    return {
        "MAPE (%)": mape(y_true, y_pred),
        "RMSE (kWh)": rmse(y_true, y_pred),
        "MAE (kWh)": mae_metric(y_true, y_pred),
    }


print("Metric functions defined.")

---
## 4. Shared plotting configuration

Plot style, one colour per gating method, and the display names used for the day-type clusters in the figures (`DAY_TYPE_NAMES`). These names are fixed labels for cluster indices 0–3; they are only used for axis labels and printed summaries.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Publication-quality settings
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("Set2", 8)
METHOD_COLORS = {
    "Hard Gating": PALETTE[0],
    "Soft (Distance)": PALETTE[1],
    "Soft (Probability)": PALETTE[2],
    "Adaptive": PALETTE[3],
}

DAY_TYPE_NAMES = {0: "Normal weekday", 1: "Weekend", 2: "Holiday", 3: "Bridge/Atypical"}

print("Plotting configuration ready.")

---
## 5. Hard gating (mandatory baseline)

Hard gating is the simplest strategy: the day classifier and the week classifier each predict one class for the observation, and the forecast is produced by the single expert of the predicted `(type_day, type_week)` combination. This is equivalent to a standard hierarchical pipeline of classification followed by regression. If the predicted combination has no expert, the forecast is the mean of the predictions of all experts (fallback).

**Advantages**: simple, fast, deterministic.

**Drawback**: classification errors are not mitigated. If the classifier assigns an observation to the wrong cluster, the forecast comes from an unsuitable expert with no possibility of correction.

The next cell recomputes the chronological 80/20 split of Section 2.5 (same arrays); all gating mechanisms are evaluated on the resulting test set.

In [ ]:
X_all = df_features[FEATURE_COLS].values
y_all = df_features["demand"].values
y_day = df_features["type_day"].values.astype(int)
y_week = df_features["type_week"].values.astype(int)

# Chronological train/test split: last 20% of rows for testing
split_idx = int(len(df_features) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
y_day_train, y_day_test = y_day[:split_idx], y_day[split_idx:]
y_week_train, y_week_test = y_week[:split_idx], y_week[split_idx:]

In [ ]:
# ============================================================
# Hard gating — classify, select a single expert and predict
# ============================================================

def predict_hard_gating(X, classifier_day, classifier_week, experts):
    """Hard gating: one expert per observation, chosen from the classifier outputs.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    classifier_day, classifier_week : fitted scikit-learn classifiers
    experts : dict[(int, int), model]

    Returns
    -------
    predictions : np.ndarray, shape (n_samples,)
    metadata : dict with routing information
    """
    pred_day = classifier_day.predict(X)
    pred_week = classifier_week.predict(X)
    predictions = np.full(len(X), np.nan)
    expert_used = []

    for i in range(len(X)):
        key = (int(pred_day[i]), int(pred_week[i]))
        if key in experts:
            predictions[i] = experts[key].predict(X[[i]])[0]
            expert_used.append(key)
        else:
            # Fallback: mean of all experts
            preds_all = [exp.predict(X[[i]])[0] for exp in experts.values()]
            predictions[i] = np.mean(preds_all)
            expert_used.append("fallback")

    n_fallback = sum(1 for e in expert_used if e == "fallback")
    return predictions, {
        "expert_used": expert_used,
        "n_fallback": n_fallback,
        "pred_day": pred_day,
        "pred_week": pred_week,
    }


# Run hard gating on the test set
print("Running hard gating on the test set...")
y_pred_hard, meta_hard = predict_hard_gating(
    X_test, classifier_day, classifier_week, experts
)

metrics_hard = compute_all_metrics(y_test, y_pred_hard)
print(f"\n--- Hard gating results ---")
for k, v in metrics_hard.items():
    print(f"  {k}: {v:.4f}")
print(f"  Fallback predictions: {meta_hard['n_fallback']:,} / {len(y_test):,}")

# Store for the comparison
all_results = []
all_results.append({
    "method": "Hard Gating",
    "params": "-",
    **metrics_hard,
    "pct_hard": 100.0,
    "pct_soft": 0.0,
})

---
## 6. Distance-based soft gating

Instead of assigning the observation to a single expert, soft gating forms a **weighted combination** of several experts. In this variant the weights are derived from the distance between the observation and each cluster centroid of Notebook 03:

$$w_k = \text{softmax}\left(\frac{-d_k}{\tau}\right) = \frac{\exp(-d_k / \tau)}{\sum_j \exp(-d_j / \tau)}$$

where $d_k$ is the Euclidean distance between the observation and centroid $k$, and the temperature $\tau$ controls how concentrated the weights are:

- **Low $\tau$** (e.g. 0.1): the weight concentrates on the nearest centroid (approaches hard gating).
- **High $\tau$** (e.g. 10.0): the weights spread more evenly across experts (approaches a uniform average).

Implementation as coded below:

- Day and week weights are computed separately, $w^{\text{day}}_i$ and $w^{\text{week}}_j$, and the weight of expert $(i, j)$ is their product $w^{\text{day}}_i\,w^{\text{week}}_j$, renormalised over the experts that exist.
- Because the centroids live in the space of daily/weekly demand profiles, not in the feature space, each observation is represented by its **first $D$ feature values** (with $D$ the centroid dimension; zero-padded if fewer features are available) as an approximation of its profile. With the feature order of Section 2.4, these are the calendar features followed by the demand lags (in the sorted order of their column names).

The temperature is swept over `TAU_SEARCH` and the value with the lowest test MAPE is retained.

In [ ]:
# ============================================================
# Soft gating — distance-based, with temperature
# ============================================================

# Load the centroids
centroids_day_path = Path(DATA_DIR) / "clustering" / "centroids_day.pkl"
centroids_week_path = Path(DATA_DIR) / "clustering" / "centroids_week.pkl"

with open(centroids_day_path, "rb") as f:
    centroids_day_dict = pickle.load(f)
with open(centroids_week_path, "rb") as f:
    centroids_week_dict = pickle.load(f)

# Extract the centroid arrays from the dictionaries
centroids_day = centroids_day_dict['centroids']
centroids_week = centroids_week_dict['centroids']

print(f"Centroids loaded: day={centroids_day.shape}, week={centroids_week.shape}")

K_DAY = centroids_day.shape[0]
K_WEEK = centroids_week.shape[0]


def compute_distance_weights(X, centroids_day, centroids_week, tau):
    """Compute soft-gating weights from the distances to the cluster centroids.

    For each observation:
      - distance to every day centroid (the first dim_day features are used as a proxy of the daily profile)
      - distance to every week centroid (the first dim_week features are used as a proxy of the weekly profile)
      - joint weight: w_{ij} = softmax(-d_day_i/tau) * softmax(-d_week_j/tau)

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    centroids_day : np.ndarray, shape (K_day, centroid_dim)
    centroids_week : np.ndarray, shape (K_week, centroid_dim)
    tau : float

    Returns
    -------
    weights_day : np.ndarray, shape (n_samples, K_day)
    weights_week : np.ndarray, shape (n_samples, K_week)
    """
    n = X.shape[0]
    dim_day = centroids_day.shape[1]
    dim_week = centroids_week.shape[1]

    # Use the first `dim` features as a proxy for the profile
    X_proj_day = X[:, :min(dim_day, X.shape[1])]
    X_proj_week = X[:, :min(dim_week, X.shape[1])]

    # Zero-pad if needed
    if X_proj_day.shape[1] < dim_day:
        X_proj_day = np.hstack([
            X_proj_day,
            np.zeros((n, dim_day - X_proj_day.shape[1]))
        ])
    if X_proj_week.shape[1] < dim_week:
        X_proj_week = np.hstack([
            X_proj_week,
            np.zeros((n, dim_week - X_proj_week.shape[1]))
        ])

    # Distances: (n_samples, K)
    dist_day = np.linalg.norm(
        X_proj_day[:, np.newaxis, :] - centroids_day[np.newaxis, :, :], axis=2
    )
    dist_week = np.linalg.norm(
        X_proj_week[:, np.newaxis, :] - centroids_week[np.newaxis, :, :], axis=2
    )

    # Soft-gating weights: w_k = softmax(-d_k / tau)
    weights_day = scipy_softmax(-dist_day / tau, axis=1)
    weights_week = scipy_softmax(-dist_week / tau, axis=1)

    return weights_day, weights_week


def predict_soft_distance(X, centroids_day, centroids_week, experts,
                          day_labels, week_labels, tau):
    """Predict with distance-based soft gating."""
    weights_day, weights_week = compute_distance_weights(
        X, centroids_day, centroids_week, tau
    )
    predictions = np.zeros(len(X))

    for i in range(len(X)):
        pred_i = 0.0
        total_w = 0.0
        for di, d_label in enumerate(day_labels):
            for wi, w_label in enumerate(week_labels):
                key = (d_label, w_label)
                if key in experts:
                    w_joint = weights_day[i, di] * weights_week[i, wi]
                    pred_i += w_joint * experts[key].predict(X[[i]])[0]
                    total_w += w_joint
        predictions[i] = pred_i / total_w if total_w > 0 else np.nan

    return predictions


# Sweep over tau
print("Sweeping the temperature tau for distance-based soft gating...")
print(f"  tau values: {TAU_SEARCH}")
print()

results_distance = []
for tau in TAU_SEARCH:
    y_pred_dist = predict_soft_distance(
        X_test, centroids_day, centroids_week, experts,
        day_labels, week_labels, tau
    )
    metrics_dist = compute_all_metrics(y_test, y_pred_dist)
    results_distance.append({"tau": tau, **metrics_dist})
    all_results.append({
        "method": "Soft (Distance)",
        "params": f"tau={tau}",
        **metrics_dist,
        "pct_hard": 0.0,
        "pct_soft": 100.0,
    })
    print(f"  tau={tau:5.1f} => MAPE={metrics_dist['MAPE (%)']:6.3f}%  "
          f"RMSE={metrics_dist['RMSE (kWh)']:.2f} kWh")

# Best tau
df_dist = pd.DataFrame(results_distance)
best_tau_idx = df_dist["MAPE (%)"].idxmin()
best_tau = df_dist.loc[best_tau_idx, "tau"]
print(f"\nBest tau: {best_tau} (MAPE = {df_dist.loc[best_tau_idx, 'MAPE (%)']:.4f}%)")

In [ ]:
# Plot the tau sweep
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_dist["tau"], df_dist["MAPE (%)"], "o-", color=PALETTE[1],
        linewidth=2, markersize=8, label="Soft (Distance)")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard Gating ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axvline(best_tau, color="gray", linestyle=":", alpha=0.6)
ax.annotate(f"Best: tau={best_tau}",
            xy=(best_tau, df_dist.loc[best_tau_idx, "MAPE (%)"]),
            xytext=(best_tau * 1.5, df_dist["MAPE (%)"].max() * 0.98),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10)
ax.set_xlabel("Temperature (tau)")
ax.set_ylabel("MAPE (%)")
ax.set_title("Distance-Based Soft Gating: MAPE vs Temperature")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_soft_distance_tau_sweep.png", bbox_inches="tight")
plt.show()

---
## 7. Probability-based soft gating

An alternative to distance-based weights is to use the **classifiers' own probability estimates** as gate weights. Since there are two separate classifiers (day and week), the joint probability is computed under an independence assumption:

$$P(\text{day}=i, \text{week}=j \mid x) = P(\text{day}=i \mid x) \times P(\text{week}=j \mid x)$$

The forecast is a weighted combination over all cluster pairs whose joint probability exceeds a minimum value $P_{\min}$ (`MIN_PROB` = 0.05), which avoids spending computation on negligible combinations:

$$\hat{y} = \frac{\sum_{i,j} P(i,j \mid x) \cdot f_{ij}(x)}{\sum_{i,j} P(i,j \mid x)}$$

where the sums run over the pairs with $P(i,j \mid x) > P_{\min}$ that have an expert $f_{ij}$. If no pair qualifies, the mean of all experts is used (fallback).

The figure produced below shows (left) how many experts take part in each forecast and (right) the distribution, over the test set, of the confidence $\min(\max_i P(\text{day}=i\mid x), \max_j P(\text{week}=j\mid x))$ used by adaptive gating in Section 8; the dashed line marks the class default threshold of 0.75. File: `06_soft_probability_analysis.png`.

In [ ]:
# ============================================================
# Soft gating — based on the classifier probabilities
# ============================================================

def predict_soft_probability(X, classifier_day, classifier_week, experts,
                             min_prob=0.05):
    """Soft gating with the joint probabilities of the day and week classifiers.

    Parameters
    ----------
    X : np.ndarray
    classifier_day, classifier_week : fitted classifiers with predict_proba
    experts : dict[(int,int), model]
    min_prob : float
        Minimum joint probability for an expert to be included.

    Returns
    -------
    predictions : np.ndarray
    metadata : dict
    """
    probs_day = classifier_day.predict_proba(X)
    probs_week = classifier_week.predict_proba(X)
    day_classes = classifier_day.classes_
    week_classes = classifier_week.classes_

    predictions = np.zeros(len(X))
    n_experts_used = []

    for i in range(len(X)):
        pred_i = 0.0
        total_w = 0.0
        n_exp = 0

        for di, d_class in enumerate(day_classes):
            for wi, w_class in enumerate(week_classes):
                p_joint = probs_day[i, di] * probs_week[i, wi]
                if p_joint > min_prob:
                    key = (int(d_class), int(w_class))
                    if key in experts:
                        pred_i += p_joint * experts[key].predict(X[[i]])[0]
                        total_w += p_joint
                        n_exp += 1

        if total_w > 0:
            predictions[i] = pred_i / total_w
        else:
            # Fallback: mean of all experts
            preds_all = [exp.predict(X[[i]])[0] for exp in experts.values()]
            predictions[i] = np.mean(preds_all)
            n_exp = len(experts)

        n_experts_used.append(n_exp)

    return predictions, {
        "n_experts_used": n_experts_used,
        "avg_experts": np.mean(n_experts_used),
        "probs_day": probs_day,
        "probs_week": probs_week,
    }


print("Running probability-based soft gating on the test set...")
y_pred_prob, meta_prob = predict_soft_probability(
    X_test, classifier_day, classifier_week, experts, min_prob=MIN_PROB
)

metrics_prob = compute_all_metrics(y_test, y_pred_prob)
print(f"\n--- Soft gating (probability) results ---")
for k, v in metrics_prob.items():
    print(f"  {k}: {v:.4f}")
print(f"  Mean number of experts used per observation: {meta_prob['avg_experts']:.2f}")

all_results.append({
    "method": "Soft (Probability)",
    "params": f"min_prob={MIN_PROB}",
    **metrics_prob,
    "pct_hard": 0.0,
    "pct_soft": 100.0,
})

In [ ]:
# Distribution of the number of experts used
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of the number of experts used
ax = axes[0]
n_exp_array = np.array(meta_prob["n_experts_used"])
unique_vals, counts = np.unique(n_exp_array, return_counts=True)
ax.bar(unique_vals, counts / len(n_exp_array) * 100, color=PALETTE[2], edgecolor="white")
ax.set_xlabel("Number of experts in ensemble")
ax.set_ylabel("Percentage of observations (%)")
ax.set_title("Distribution of Expert Count in Soft (Probability) Gating")
for v, c in zip(unique_vals, counts / len(n_exp_array) * 100):
    ax.text(v, c + 0.5, f"{c:.1f}%", ha="center", fontsize=9)

# Right: distribution of the minimum classifier confidence
ax = axes[1]
max_conf_day = meta_prob["probs_day"].max(axis=1)
max_conf_week = meta_prob["probs_week"].max(axis=1)
min_conf = np.minimum(max_conf_day, max_conf_week)
ax.hist(min_conf, bins=50, color=PALETTE[3], edgecolor="white", alpha=0.8)
ax.set_xlabel("Minimum confidence (min of day, week)")
ax.set_ylabel("Count")
ax.set_title("Classifier Confidence Distribution on Test Set")
ax.axvline(0.75, color="red", linestyle="--", label="Default threshold (0.75)")
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_soft_probability_analysis.png", bbox_inches="tight")
plt.show()

---
## 8. Confidence-based adaptive gating

Adaptive gating is the **central contribution** of this notebook. It combines the efficiency of hard gating with the robustness of soft gating by switching between the two according to the classifiers' confidence.

### 8.1 Algorithm

For each observation $x$:

1. Compute the classifier probabilities $P_{\text{day}}$ and $P_{\text{week}}$.
2. Compute the confidence $c = \min(\max(P_{\text{day}}), \max(P_{\text{week}}))$.
3. If $c > \theta$ (threshold): **hard gating** — use the expert of the most probable `(day, week)` pair.
4. If $c \leq \theta$: **soft gating** — combine experts weighted by the joint probability, as in Section 7.

The threshold $\theta$ is a hyperparameter that controls the trade-off:

- **High $\theta$** (e.g. 0.95): most observations use soft gating (conservative, more robust).
- **Low $\theta$** (e.g. 0.5): most observations use hard gating (aggressive, trusts the classifier).

### 8.2 Motivation

In electricity demand forecasting the classifier is expected to be reliable on "typical" days (normal weekdays, regular weekends), but it can fail on, for example:

- **Holidays** that fall on unusual days of the week;
- **Bridge holidays** (Monday holidays that create a long weekend);
- **Seasonal transitions** (dry → rainy season and vice versa);
- **Special events** (elections, sporting events, etc.).

For such ambiguous cases, soft gating hedges the risk by combining several experts instead of committing to a single, possibly wrong, expert. Whether this lowers the error on the test set is assessed empirically in Section 9.

In [ ]:
# ============================================================
# Adaptive gating — confidence-based switching
# ============================================================

class AdaptiveGating:
    """Adaptive gating mechanism for Mixture-of-Experts forecasting.

    Routes observations with hard gating when the classifier confidence
    is high, and falls back to probability-weighted soft gating when the
    confidence is low.

    Parameters
    ----------
    classifier_day : sklearn classifier
        Fitted classifier for type_day with predict_proba().
    classifier_week : sklearn classifier
        Fitted classifier for type_week with predict_proba().
    experts : dict[tuple[int, int], regressor]
        Mapping (cluster_day, cluster_week) -> fitted regression model.
    threshold : float
        Confidence threshold for switching between hard and soft gating.
        min(conf_day, conf_week) > threshold => Hard Gating.
    min_prob : float
        Minimum joint probability for an expert to be included in soft gating.
    """

    def __init__(self, classifier_day, classifier_week, experts,
                 threshold=0.75, min_prob=0.05):
        self.classifier_day = classifier_day
        self.classifier_week = classifier_week
        self.experts = experts
        self.threshold = threshold
        self.min_prob = min_prob

    def predict(self, X):
        """Generate predictions with adaptive gating.

        Returns
        -------
        predictions : np.ndarray
        metadata : dict
            - gating_mode: list of 'hard'/'soft', one per observation
            - confidence: confidence score per observation
            - expert_used: expert key or 'soft_ensemble'
        """
        X = np.asarray(X)
        n = len(X)

        probs_day = self.classifier_day.predict_proba(X)
        probs_week = self.classifier_week.predict_proba(X)
        day_classes = self.classifier_day.classes_
        week_classes = self.classifier_week.classes_

        max_conf_day = probs_day.max(axis=1)
        max_conf_week = probs_week.max(axis=1)
        min_confidence = np.minimum(max_conf_day, max_conf_week)

        predictions = np.zeros(n)
        gating_modes = []
        expert_used = []

        for i in range(n):
            if min_confidence[i] > self.threshold:
                # --- HARD GATING ---
                cd = day_classes[probs_day[i].argmax()]
                cw = week_classes[probs_week[i].argmax()]
                key = (int(cd), int(cw))

                if key in self.experts:
                    predictions[i] = self.experts[key].predict(X[[i]])[0]
                else:
                    predictions[i] = self._fallback(X[[i]])
                    key = "fallback"

                gating_modes.append("hard")
                expert_used.append(key)
            else:
                # --- SOFT GATING ---
                pred_i = 0.0
                total_w = 0.0

                for di, d_class in enumerate(day_classes):
                    for wi, w_class in enumerate(week_classes):
                        p_joint = probs_day[i, di] * probs_week[i, wi]
                        if p_joint > self.min_prob:
                            key_ij = (int(d_class), int(w_class))
                            if key_ij in self.experts:
                                pred_i += p_joint * self.experts[key_ij].predict(X[[i]])[0]
                                total_w += p_joint

                if total_w > 0:
                    predictions[i] = pred_i / total_w
                else:
                    predictions[i] = self._fallback(X[[i]])

                gating_modes.append("soft")
                expert_used.append("soft_ensemble")

        n_hard = sum(1 for m in gating_modes if m == "hard")
        n_soft = n - n_hard

        return predictions, {
            "gating_mode": gating_modes,
            "confidence": min_confidence,
            "expert_used": expert_used,
            "n_hard": n_hard,
            "n_soft": n_soft,
            "pct_hard": n_hard / n * 100,
            "pct_soft": n_soft / n * 100,
        }

    def _fallback(self, x):
        """Fallback: mean of all available experts."""
        if not self.experts:
            return 0.0
        preds = [exp.predict(x)[0] for exp in self.experts.values()]
        return float(np.mean(preds))


print("AdaptiveGating class defined.")
print(f"Threshold search grid: {THRESHOLD_SEARCH}")

### 8.3 Threshold sweep

Adaptive gating is evaluated on the test set for every threshold in `THRESHOLD_SEARCH`; the threshold with the lowest test MAPE is retained as `best_threshold`. The figure shows MAPE versus threshold (with hard gating and probability-based soft gating as references) and the share of observations routed by hard and by soft gating. File: `06_adaptive_threshold_sweep.png`.

In [ ]:
# ============================================================
# Sweep the confidence threshold of adaptive gating
# ============================================================

print("Sweeping the confidence threshold for adaptive gating...")
print(f"{'Threshold':>10} {'MAPE (%)':>10} {'RMSE (kWh)':>12} {'% Hard':>8} {'% Soft':>8}")
print("-" * 55)

results_adaptive = []
adaptive_predictions = {}  # stored for later analysis
adaptive_metadata = {}

for threshold in THRESHOLD_SEARCH:
    gating = AdaptiveGating(
        classifier_day=classifier_day,
        classifier_week=classifier_week,
        experts=experts,
        threshold=threshold,
        min_prob=MIN_PROB,
    )

    y_pred_adapt, meta_adapt = gating.predict(X_test)
    metrics_adapt = compute_all_metrics(y_test, y_pred_adapt)

    adaptive_predictions[threshold] = y_pred_adapt
    adaptive_metadata[threshold] = meta_adapt

    results_adaptive.append({
        "threshold": threshold,
        **metrics_adapt,
        "pct_hard": meta_adapt["pct_hard"],
        "pct_soft": meta_adapt["pct_soft"],
    })

    all_results.append({
        "method": "Adaptive",
        "params": f"threshold={threshold}",
        **metrics_adapt,
        "pct_hard": meta_adapt["pct_hard"],
        "pct_soft": meta_adapt["pct_soft"],
    })

    print(f"{threshold:>10.2f} {metrics_adapt['MAPE (%)']:>10.4f} "
          f"{metrics_adapt['RMSE (kWh)']:>12.2f} "
          f"{meta_adapt['pct_hard']:>7.1f}% {meta_adapt['pct_soft']:>7.1f}%")

# Best threshold
df_adaptive = pd.DataFrame(results_adaptive)
best_thresh_idx = df_adaptive["MAPE (%)"].idxmin()
best_threshold = df_adaptive.loc[best_thresh_idx, "threshold"]
print(f"\nBest threshold: {best_threshold}")
print(f"  MAPE: {df_adaptive.loc[best_thresh_idx, 'MAPE (%)']:.4f}%")
print(f"  Hard/soft split: {df_adaptive.loc[best_thresh_idx, 'pct_hard']:.1f}% / "
      f"{df_adaptive.loc[best_thresh_idx, 'pct_soft']:.1f}%")

In [ ]:
# Plot the adaptive-gating threshold sweep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MAPE vs threshold
ax = axes[0]
ax.plot(df_adaptive["threshold"], df_adaptive["MAPE (%)"], "o-",
        color=PALETTE[3], linewidth=2, markersize=8, label="Adaptive")
ax.axhline(metrics_hard["MAPE (%)"], color=PALETTE[0], linestyle="--",
           linewidth=1.5, label=f"Hard ({metrics_hard['MAPE (%)']:.3f}%)")
ax.axhline(metrics_prob["MAPE (%)"], color=PALETTE[2], linestyle="-.",
           linewidth=1.5, label=f"Soft-Prob ({metrics_prob['MAPE (%)']:.3f}%)")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6)
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("MAPE (%)")
ax.set_title("Adaptive Gating: MAPE vs Confidence Threshold")
ax.legend(loc="best")

# Right: Hard/Soft percentage vs threshold
ax = axes[1]
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"],
                alpha=0.4, color=PALETTE[0], label="% Hard gating")
ax.fill_between(df_adaptive["threshold"], df_adaptive["pct_hard"], 100,
                alpha=0.4, color=PALETTE[2], label="% Soft gating")
ax.axvline(best_threshold, color="gray", linestyle=":", alpha=0.6,
           label=f"Best threshold ({best_threshold})")
ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Percentage of Observations (%)")
ax.set_title("Hard vs Soft Gating Split by Threshold")
ax.set_ylim(0, 105)
ax.legend(loc="center left")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_adaptive_threshold_sweep.png", bbox_inches="tight")
plt.show()

---
## 9. Comprehensive comparison and analysis

All gating strategies are now compared along several dimensions:

1. **Global metrics**: overall MAPE, RMSE and MAE on the test set.
2. **By hour of day**: where does soft gating help most?
3. **By day type**: holidays, bridge holidays, weekends, etc.
4. **Confidence analysis**: how do hard-gated and soft-gated observations differ?

For the per-hour and per-day-type analyses, each method is represented by its lowest-MAPE configuration (best $\tau$ for distance-based soft gating, best $\theta$ for adaptive gating).

### 9.1 Summary table

The table lists every configuration evaluated and the best configuration of each method. As implemented, the configuration reported as the *selected overall configuration* (`overall_best`, later written to the JSON file of Section 10) is the one ranked **ninth** by test MAPE among all configurations (`nsmallest(9).index[8]`); the line that selects the minimum-MAPE configuration is commented out in the code.

In [ ]:
# ============================================================
# Comparison summary table
# ============================================================

df_results = pd.DataFrame(all_results)

# Full table
print("=" * 90)
print("GATING STRATEGY COMPARISON — ALL CONFIGURATIONS")
print("=" * 90)
display_cols = ["method", "params", "MAPE (%)", "RMSE (kWh)", "MAE (kWh)", "pct_hard", "pct_soft"]
print(df_results[display_cols].to_string(index=False, float_format="{:.4f}".format))

# Best configuration per method
print("\n" + "=" * 70)
print("BEST CONFIGURATION PER METHOD")
print("=" * 70)
best_per_method = df_results.loc[df_results.groupby("method")["MAPE (%)"].idxmin()]
print(best_per_method[display_cols].to_string(index=False, float_format="{:.4f}".format))

# Selected overall configuration (9th-lowest MAPE; the minimum-MAPE line is commented out)
#overall_best_idx = df_results["MAPE (%)"].idxmin()
overall_best_idx = df_results["MAPE (%)"].nsmallest(9).index[8]
overall_best = df_results.loc[overall_best_idx]
print(f"\nSELECTED OVERALL CONFIGURATION: {overall_best['method']} ({overall_best['params']}) "
      f"=> MAPE = {overall_best['MAPE (%)']:.4f}%")

### 9.2 MAPE by hour of day

Test MAPE of each method for each hour of the day. File: `06_mape_by_hour_comparison.png`.

In [ ]:
# ============================================================
# MAPE by hour of day for each gating method
# ============================================================

# Get predictions for best of each method
# Hard: already have y_pred_hard
# Soft distance: use best tau
y_pred_dist_best = predict_soft_distance(
    X_test, centroids_day, centroids_week, experts,
    day_labels, week_labels, best_tau
)
# Soft probability: already have y_pred_prob
# Adaptive: use best threshold
y_pred_adapt_best = adaptive_predictions[best_threshold]

# Compute MAPE by hour

df_test['datetime'] = pd.to_datetime(df_test['datetime'])
hours_test = df_test['datetime'].dt.hour.values

def mape_by_group(y_true, y_pred, groups, group_values):
    """Compute MAPE for each group."""
    result = {}
    for g in group_values:
        mask = groups == g
        if mask.sum() > 0:
            result[g] = mape(y_true[mask], y_pred[mask])
        else:
            result[g] = np.nan
    return result

hours_range = np.arange(24)
mape_hour_hard = mape_by_group(y_test, y_pred_hard, hours_test, hours_range)
mape_hour_dist = mape_by_group(y_test, y_pred_dist_best, hours_test, hours_range)
mape_hour_prob = mape_by_group(y_test, y_pred_prob, hours_test, hours_range)
mape_hour_adapt = mape_by_group(y_test, y_pred_adapt_best, hours_test, hours_range)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(24)
width = 0.2

ax.bar(x - 1.5*width, [mape_hour_hard[h] for h in hours_range], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_hour_dist[h] for h in hours_range], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_hour_prob[h] for h in hours_range], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_hour_adapt[h] for h in hours_range], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Hour of Day")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE by Hour of Day — Gating Strategy Comparison")
ax.set_xticks(x)
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_by_hour_comparison.png", bbox_inches="tight")
plt.show()

### 9.3 MAPE by day type

Test MAPE of each method for each day-type cluster (true `type_day` of the test observations), with the number of test observations of each type. File: `06_mape_by_day_type_comparison.png`.

In [ ]:
# ============================================================
# MAPE by day type for each gating method
# ============================================================

type_day_test = df_test["type_day"].values

mape_day_hard = mape_by_group(y_test, y_pred_hard, type_day_test, day_labels)
mape_day_dist = mape_by_group(y_test, y_pred_dist_best, type_day_test, day_labels)
mape_day_prob = mape_by_group(y_test, y_pred_prob, type_day_test, day_labels)
mape_day_adapt = mape_by_group(y_test, y_pred_adapt_best, type_day_test, day_labels)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(day_labels))
width = 0.2

ax.bar(x - 1.5*width, [mape_day_hard[d] for d in day_labels], width,
       label="Hard Gating", color=METHOD_COLORS["Hard Gating"], edgecolor="white")
ax.bar(x - 0.5*width, [mape_day_dist[d] for d in day_labels], width,
       label=f"Soft (Distance, tau={best_tau})", color=METHOD_COLORS["Soft (Distance)"],
       edgecolor="white")
ax.bar(x + 0.5*width, [mape_day_prob[d] for d in day_labels], width,
       label="Soft (Probability)", color=METHOD_COLORS["Soft (Probability)"],
       edgecolor="white")
ax.bar(x + 1.5*width, [mape_day_adapt[d] for d in day_labels], width,
       label=f"Adaptive (thr={best_threshold})", color=METHOD_COLORS["Adaptive"],
       edgecolor="white")

ax.set_xlabel("Day Type")
ax.set_ylabel("MAPE (%)")
ax.set_title("MAPE by Day Type — Gating Strategy Comparison")
ax.set_xticks(x)
ax.set_xticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels],
                   rotation=15, ha="right")
ax.legend(loc="upper right")

# Add count annotations
for i, d in enumerate(day_labels):
    n = (type_day_test == d).sum()
    ax.text(i, ax.get_ylim()[1] * 0.02, f"n={n:,}",
            ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_by_day_type_comparison.png", bbox_inches="tight")
plt.show()

### 9.4 Where does soft gating help?

For the best adaptive configuration, the test observations are split into those routed by hard gating and those routed by soft gating. For the soft-gated (low-confidence) observations, the MAPE obtained by adaptive gating is compared with the MAPE that pure hard gating obtains on the same observations, and the soft-gated observations are broken down by day type.

In [ ]:
# ============================================================
# Analysis — where does soft gating improve the forecast?
# ============================================================

best_meta = adaptive_metadata[best_threshold]
gating_modes_best = np.array(best_meta["gating_mode"])
confidence_best = best_meta["confidence"]

# Split errors by gating mode
hard_mask = gating_modes_best == "hard"
soft_mask = gating_modes_best == "soft"

print("=" * 70)
print(f"ADAPTIVE GATING ANALYSIS (threshold={best_threshold})")
print("=" * 70)
print(f"\nTotal test observations: {len(y_test):,}")
print(f"  Hard gating: {hard_mask.sum():,} ({hard_mask.mean()*100:.1f}%)")
print(f"  Soft gating: {soft_mask.sum():,} ({soft_mask.mean()*100:.1f}%)")

if hard_mask.sum() > 0:
    print(f"\n  MAPE hard-gated: {mape(y_test[hard_mask], y_pred_adapt_best[hard_mask]):.4f}%")
if soft_mask.sum() > 0:
    print(f"  MAPE soft-gated: {mape(y_test[soft_mask], y_pred_adapt_best[soft_mask]):.4f}%")

# Counterfactual: what would hard gating have achieved on the soft-gated observations?
if soft_mask.sum() > 0:
    # Hard-gating predictions on the soft-gated observations
    y_hard_on_soft = y_pred_hard[soft_mask]
    y_adapt_on_soft = y_pred_adapt_best[soft_mask]
    y_true_soft = y_test[soft_mask]

    mape_hard_on_soft = mape(y_true_soft, y_hard_on_soft)
    mape_adapt_on_soft = mape(y_true_soft, y_adapt_on_soft)

    print(f"\n  For the {soft_mask.sum():,} soft-gated observations:")
    print(f"    Hard gating would have achieved MAPE = {mape_hard_on_soft:.4f}%")
    print(f"    Soft gating achieved MAPE            = {mape_adapt_on_soft:.4f}%")
    improvement = mape_hard_on_soft - mape_adapt_on_soft
    if improvement > 0:
        print(f"    => Soft gating IMPROVED MAPE by {improvement:.4f} pp on low-confidence observations")
    else:
        print(f"    => Soft gating was WORSE by {-improvement:.4f} pp (hard gating was better)")

# Breakdown of the soft-gated observations by day type
if soft_mask.sum() > 0:
    print("\n  Soft-gated observations by day type:")
    for d in day_labels:
        mask_d = soft_mask & (type_day_test == d)
        pct = mask_d.sum() / soft_mask.sum() * 100 if soft_mask.sum() > 0 else 0
        name = DAY_TYPE_NAMES.get(d, f"Type {d}")
        print(f"    {name}: {mask_d.sum():,} obs ({pct:.1f}%)")

### 9.5 Classifier confidence vs. forecast error

Left: absolute percentage error (APE) of the best adaptive configuration against the classifier confidence, coloured by gating mode, with a centred moving average (window of up to 500 observations sorted by confidence). Right: APE distributions of hard-gated and soft-gated observations. File: `06_confidence_vs_error_analysis.png`.

In [ ]:
# ============================================================
# Confidence vs. error scatter plot and error distributions
# ============================================================

ape_adapt = np.abs((y_test - y_pred_adapt_best) / np.where(y_test != 0, y_test, 1)) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Confidence vs APE
ax = axes[0]
scatter = ax.scatter(
    confidence_best, ape_adapt,
    c=[METHOD_COLORS["Hard Gating"] if m == "hard" else METHOD_COLORS["Soft (Probability)"]
       for m in gating_modes_best],
    alpha=0.15, s=5, rasterized=True
)
ax.axvline(best_threshold, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold = {best_threshold}")

# Add moving average trend
conf_sorted_idx = np.argsort(confidence_best)
conf_sorted = confidence_best[conf_sorted_idx]
ape_sorted = ape_adapt[conf_sorted_idx]
window = min(500, len(ape_sorted) // 10)
if window > 0:
    ape_rolling = pd.Series(ape_sorted).rolling(window, center=True).mean()
    ax.plot(conf_sorted, ape_rolling, color="black", linewidth=2, label="Moving avg")

ax.set_xlabel("Classifier Confidence (min of day, week)")
ax.set_ylabel("Absolute Percentage Error (%)")
ax.set_title("Confidence vs Prediction Error")
ax.set_ylim(0, np.percentile(ape_adapt, 99))
ax.legend()

# Right: APE distribution for hard vs soft
ax = axes[1]
if hard_mask.sum() > 0:
    ax.hist(ape_adapt[hard_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Hard Gating"], label="Hard-gated", edgecolor="white")
if soft_mask.sum() > 0:
    ax.hist(ape_adapt[soft_mask], bins=60, alpha=0.6, density=True,
            color=METHOD_COLORS["Soft (Probability)"], label="Soft-gated", edgecolor="white")
ax.set_xlabel("Absolute Percentage Error (%)")
ax.set_ylabel("Density")
ax.set_title("Error Distribution by Gating Mode")
ax.set_xlim(0, np.percentile(ape_adapt, 98))
ax.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_confidence_vs_error_analysis.png", bbox_inches="tight")
plt.show()

### 9.6 MAPE by hour and day type (best adaptive gating)

MAPE of the best adaptive configuration for each (day type, hour) cell; cells with 5 or fewer test observations are left empty. File: `06_mape_heatmap_adaptive.png`.

In [ ]:
# ============================================================
# Heatmap — MAPE by (hour, day type) for the best adaptive gating
# ============================================================

# Build MAPE matrix: rows=day_type, cols=hour
mape_matrix = np.full((len(day_labels), 24), np.nan)
for di, d in enumerate(day_labels):
    for h in range(24):
        mask = (type_day_test == d) & (hours_test == h)
        if mask.sum() > 5:  # only if enough observations
            mape_matrix[di, h] = mape(y_test[mask], y_pred_adapt_best[mask])

fig, ax = plt.subplots(figsize=(16, 4))
im = ax.imshow(mape_matrix, aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(len(day_labels)))
ax.set_yticklabels([DAY_TYPE_NAMES.get(d, f"Type {d}") for d in day_labels])
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Day Type")
ax.set_title(f"MAPE (%) Heatmap — Best Adaptive Gating (threshold={best_threshold})")

# Add text annotations
for di in range(len(day_labels)):
    for h in range(24):
        if not np.isnan(mape_matrix[di, h]):
            color = "white" if mape_matrix[di, h] > np.nanmedian(mape_matrix) else "black"
            ax.text(h, di, f"{mape_matrix[di, h]:.1f}",
                    ha="center", va="center", fontsize=6, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("MAPE (%)")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_mape_heatmap_adaptive.png", bbox_inches="tight")
plt.show()

### 9.7 Improvement of adaptive over hard gating

Difference in MAPE (percentage points, pp) between hard gating and the best adaptive configuration, by hour of day (figure) and by day type (printed); positive values mean adaptive gating is better. File: `06_improvement_adaptive_vs_hard_by_hour.png`.

In [ ]:
# ============================================================
# Improvement of adaptive over hard gating by hour
# ============================================================

improvement_by_hour = {}
for h in range(24):
    mask_h = hours_test == h
    if mask_h.sum() > 0:
        mape_h_hard = mape(y_test[mask_h], y_pred_hard[mask_h])
        mape_h_adapt = mape(y_test[mask_h], y_pred_adapt_best[mask_h])
        improvement_by_hour[h] = mape_h_hard - mape_h_adapt  # positive = adaptive is better

fig, ax = plt.subplots(figsize=(14, 5))
hours_list = sorted(improvement_by_hour.keys())
improvements = [improvement_by_hour[h] for h in hours_list]
colors = [PALETTE[3] if imp >= 0 else PALETTE[6] for imp in improvements]

ax.bar(hours_list, improvements, color=colors, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("MAPE Improvement (pp)")
ax.set_title(f"MAPE Improvement: Adaptive (thr={best_threshold}) vs Hard Gating")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])

# Annotate
for h, imp in zip(hours_list, improvements):
    if abs(imp) > 0.01:
        va = "bottom" if imp >= 0 else "top"
        ax.text(h, imp, f"{imp:+.2f}", ha="center", va=va, fontsize=7)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/06_improvement_adaptive_vs_hard_by_hour.png",
            bbox_inches="tight")
plt.show()
# Also by day type
print("\nMAPE improvement (pp) by day type (positive = adaptive is better):")
for d in day_labels:
    mask_d = type_day_test == d
    if mask_d.sum() > 0:
        mape_d_hard = mape(y_test[mask_d], y_pred_hard[mask_d])
        mape_d_adapt = mape(y_test[mask_d], y_pred_adapt_best[mask_d])
        imp = mape_d_hard - mape_d_adapt
        name = DAY_TYPE_NAMES.get(d, f"Type {d}")
        print(f"  {name}: {imp:+.4f} pp  (Hard={mape_d_hard:.4f}%, Adapt={mape_d_adapt:.4f}%)")

---
## 10. Saving the results and the selected gating configuration

The following are persisted:

1. The full comparison table and the best configuration per method (CSV, in `RESULTS_DIR`).
2. The selected gating configuration (`overall_best`, see Section 9.1), the best $\tau$ and $\theta$ of the sweeps, `MIN_PROB`, reference MAPE values and the list of expert keys (JSON, `MODELS_DIR/gating/06_best_gating_config.json`).
3. The adaptive-threshold and distance-temperature sweeps (CSV, in `RESULTS_DIR`).

The production gating hyperparameters are re-tuned in Notebook 07, which does not read this JSON file.

In [ ]:
# ============================================================
# Save the gating comparison results and the selected configuration
# ============================================================

# 1. Full comparison table
results_path = Path(RESULTS_DIR) / "06_gating_comparison.csv"
df_results.to_csv(results_path, index=False)
print(f"Full comparison saved to: {results_path}")

# 2. Best configuration per method
best_per_method_path = Path(RESULTS_DIR) / "06_best_per_method.csv"
best_per_method.to_csv(best_per_method_path, index=False)
print(f"Best per method saved to: {best_per_method_path}")

# 3. Selected gating configuration (JSON)
best_gating_config = {
    "gating_mode": "adaptive" if overall_best["method"] == "Adaptive" else overall_best["method"].lower().replace(" ", "_"),
    "best_method": overall_best["method"],
    "best_params": overall_best["params"],
    "best_mape_pct": float(overall_best["MAPE (%)"]),
    "best_rmse": float(overall_best["RMSE (kWh)"]),
    "threshold": float(best_threshold),
    "tau": float(best_tau),
    "min_prob": MIN_PROB,
    "pct_hard": float(overall_best["pct_hard"]),
    "pct_soft": float(overall_best["pct_soft"]),
    "hard_gating_mape": float(metrics_hard["MAPE (%)"]),
    "soft_distance_best_mape": float(df_dist.loc[best_tau_idx, "MAPE (%)"]),
    "soft_probability_mape": float(metrics_prob["MAPE (%)"]),
    "n_experts": len(experts),
    "expert_keys": [list(k) for k in sorted(experts.keys())],
}

config_path = Path(MODELS_DIR) / "gating" / "06_best_gating_config.json"
with open(config_path, "w") as f:
    json.dump(best_gating_config, f, indent=2)
print(f"Gating configuration saved to: {config_path}")

# 4. Adaptive-gating threshold sweep
adaptive_path = Path(RESULTS_DIR) / "06_adaptive_threshold_sweep.csv"
df_adaptive.to_csv(adaptive_path, index=False)
print(f"Adaptive sweep saved to: {adaptive_path}")

# 5. Distance-based tau sweep
distance_path = Path(RESULTS_DIR) / "06_distance_tau_sweep.csv"
df_dist.to_csv(distance_path, index=False)
print(f"Distance sweep saved to: {distance_path}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Selected gating strategy: {overall_best['method']}")
print(f"Parameters: {overall_best['params']}")
print(f"Overall MAPE: {overall_best['MAPE (%)']:.4f}%")
print(f"Hard/soft split: {overall_best['pct_hard']:.1f}% / {overall_best['pct_soft']:.1f}%")
print(f"\nResults saved in: {RESULTS_DIR}/ and {config_path.parent}/")